In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if all((candidate / name).exists() for name in ("research", "configs", "runs")):
            return candidate
    raise FileNotFoundError("C:/ronbun 내부에서 노트북을 실행하십시오.")


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 사용자 선택 영역 ---------------------------------------------------------
MODEL_NAME = "adaface"
DATASETS = ("lfw", "survface")

# 기존 셀에서 RUN_IDS로 불렀던 adaface-... 값은 model_uid입니다.
MODEL_UIDS = {
    "lfw": "adaface-4df25b75e065b0b9ed43",
    "survface": "adaface-4df25b75e065b0b9ed43",
}
# 아래 값은 실제 Step 4 workflow run_id입니다.
RUN_IDS = {
    "lfw": "20260731-R001-de27340c",
    "survface": "20260731-R001-8f94ec62",
}

CHUNKSIZE = 100_000
REBUILD_DATASET_SUMMARIES = False  # True이면 대용량 원본에서 다시 집계
VERIFY_SOURCE_SHA256 = False       # True이면 수 GB 원본 SHA-256까지 재검증
WRITE_OUTPUTS = True
OVERWRITE_COMMON_OUTPUTS = True

required_relative_paths = (
    Path("COMPLETED"),
    Path("artifacts/step2_workflow/freeze_manifest.json"),
    Path("artifacts/step2_workflow/step4_summary.json"),
    Path("artifacts/step2_workflow/paired_embedding_metrics.csv"),
    Path("artifacts/step2_workflow/retrieval_metrics.csv"),
)
candidate_rows = []
for manifest_path in sorted((PROJECT_ROOT / "runs").rglob("run_manifest.json")):
    run_dir = manifest_path.parent
    freeze_path = run_dir / "artifacts/step2_workflow/freeze_manifest.json"
    step4_path = run_dir / "artifacts/step2_workflow/step4_summary.json"
    if not freeze_path.is_file() or not step4_path.is_file():
        continue
    run_manifest = load_json(manifest_path)
    freeze = load_json(freeze_path)
    step4 = load_json(step4_path)
    model_uid = str(freeze.get("model_uid", ""))
    dataset = str(freeze.get("dataset_id", ""))
    ready = (
        run_manifest.get("status") == "completed"
        and freeze.get("fallback_free") is True
        and all((run_dir / path).exists() for path in required_relative_paths)
        and str(run_manifest.get("run_id")) == str(freeze.get("run_id"))
        and str(step4.get("run_id")) == str(freeze.get("run_id"))
    )
    paired_path = run_dir / "artifacts/step2_workflow/paired_embedding_metrics.csv"
    retrieval_path = run_dir / "artifacts/step2_workflow/retrieval_metrics.csv"
    candidate_rows.append(
        {
            "dataset": dataset,
            "model_name": model_uid.split("-", 1)[0].lower(),
            "model_uid": model_uid,
            "run_id": str(run_manifest.get("run_id", "")),
            "status": run_manifest.get("status"),
            "ready": ready,
            "mode": freeze.get("scope", {}).get("mode"),
            "data_fraction": freeze.get("scope", {}).get("data_fraction"),
            "is_paper_run": freeze.get("scope", {}).get("is_paper_run"),
            "selected_samples": step4.get("selected_samples"),
            "paired_rows": step4.get("paired_rows"),
            "retrieval_rows": step4.get("retrieval_rows"),
            "source_gib": round(
                sum(
                    path.stat().st_size if path.exists() else 0
                    for path in (paired_path, retrieval_path)
                )
                / (1024**3),
                3,
            ),
            "source_branch": run_manifest.get("git", {}).get("branch"),
            "source_commit": run_manifest.get("git", {}).get("commit"),
            "run_dir": run_dir.relative_to(PROJECT_ROOT).as_posix(),
        }
    )

EXPERIMENT_CANDIDATES = pd.DataFrame.from_records(candidate_rows)
if EXPERIMENT_CANDIDATES.empty:
    raise FileNotFoundError("완료된 Step 4 후보를 조회할 수 없습니다.")
EXPERIMENT_CANDIDATES = EXPERIMENT_CANDIDATES.sort_values(
    ["dataset", "model_name", "run_id"]
).reset_index(drop=True)
display(EXPERIMENT_CANDIDATES)

RUN_SELECTORS = {}
try:
    import ipywidgets as widgets

    for dataset in DATASETS:
        choices = EXPERIMENT_CANDIDATES.loc[
            EXPERIMENT_CANDIDATES["ready"]
            & EXPERIMENT_CANDIDATES["dataset"].eq(dataset)
            & EXPERIMENT_CANDIDATES["model_name"].eq(MODEL_NAME)
            & EXPERIMENT_CANDIDATES["model_uid"].eq(MODEL_UIDS[dataset])
        ]
        options = [
            (
                f"{row.run_id} | {row.model_uid} | {row.mode} "
                f"p={row.data_fraction}",
                row.run_id,
            )
            for row in choices.itertuples(index=False)
        ]
        values = {value for _, value in options}
        if RUN_IDS[dataset] not in values:
            raise ValueError(
                f"{dataset}: configured run_id {RUN_IDS[dataset]!r}가 "
                "조회된 완료 후보에 없습니다."
            )
        RUN_SELECTORS[dataset] = widgets.Dropdown(
            options=options,
            value=RUN_IDS[dataset],
            description=dataset,
            layout=widgets.Layout(width="900px"),
            style={"description_width": "100px"},
        )
    display(widgets.VBox(list(RUN_SELECTORS.values())))
    print("드롭다운을 변경한 뒤 다음 셀부터 실행하십시오.")
except ImportError:
    print("ipywidgets가 없어 RUN_IDS 설정값을 사용합니다.")
    display(pd.Series(RUN_IDS, name="selected_run_id").to_frame())


# LFW + SurvFace compact summary 생성과 공통 시각화

완료된 fallback-free Step 4 실행만 선택하여 다음 산출물을 만듭니다.

- 데이터셋별 `compression_summary.csv`: compression family/profile당 1행
- 데이터셋별 `retrieval_summary.csv`: family/profile/threshold policy당 1행
- 공통 `compression_summary_all.csv`: 위 두 표를 검증된 provenance key로 결합한 최종 비교표

이 노트북은 압축 또는 검색 실험을 다시 수행하지 않습니다. 대용량 원본 ledger는 읽기 전용으로 집계하며 공통 출력에 복제하지 않습니다.


In [ ]:
from scripts.generate_step4_compact_summaries import generate
from research.runtime.hashing import sha256_file

SELECTED_RUN_IDS = {
    dataset: RUN_SELECTORS[dataset].value if RUN_SELECTORS else RUN_IDS[dataset]
    for dataset in DATASETS
}

selected_rows = []
SELECTED_RUNS = {}
for dataset in DATASETS:
    matches = EXPERIMENT_CANDIDATES.loc[
        EXPERIMENT_CANDIDATES["ready"]
        & EXPERIMENT_CANDIDATES["dataset"].eq(dataset)
        & EXPERIMENT_CANDIDATES["model_name"].eq(MODEL_NAME)
        & EXPERIMENT_CANDIDATES["model_uid"].eq(MODEL_UIDS[dataset])
        & EXPERIMENT_CANDIDATES["run_id"].eq(SELECTED_RUN_IDS[dataset])
    ]
    if len(matches) != 1:
        raise ValueError(
            f"{dataset}: 선택 조건과 일치하는 완료 실행이 1개가 아닙니다: {len(matches)}"
        )
    row = matches.iloc[0]
    run_dir = PROJECT_ROOT / str(row["run_dir"])
    SELECTED_RUNS[dataset] = run_dir
    selected_rows.append(row)

selected_run_table = pd.DataFrame(selected_rows).reset_index(drop=True)
if set(selected_run_table["model_name"]) != {MODEL_NAME}:
    raise ValueError("선택 실행의 model_name이 MODEL_NAME과 일치하지 않습니다.")
display(selected_run_table)


## 1. 데이터셋별 compact summary 생성·검증

기존 summary가 있으면 output hash와 선택한 dataset/model/run 계보를 검증해 재사용합니다. 없거나 `REBUILD_DATASET_SUMMARIES=True`이면 `scripts/generate_step4_compact_summaries.py`가 원본 CSV를 chunk 단위로 읽어 생성합니다.


In [ ]:
def resolve_manifest_path(recorded_path: str) -> Path:
    path = Path(recorded_path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def validate_summary_bundle(
    summary_dir: Path,
    *,
    dataset: str,
    run_dir: Path,
) -> dict:
    manifest_path = summary_dir / "summary_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    payload = load_json(manifest_path)
    expected_identity = {
        "artifact_type": "step4_compact_summaries",
        "dataset_id": dataset,
        "model_uid": MODEL_UIDS[dataset],
        "run_id": SELECTED_RUN_IDS[dataset],
        "source_run_status": "completed",
    }
    mismatches = {
        key: (payload.get(key), expected)
        for key, expected in expected_identity.items()
        if payload.get(key) != expected
    }
    if mismatches:
        raise ValueError(f"{dataset}: summary identity mismatch: {mismatches}")

    source_manifest = payload.get("source_files", {}).get("run_manifest.json", {})
    recorded_source = resolve_manifest_path(str(source_manifest.get("path", "")))
    if recorded_source.resolve() != (run_dir / "run_manifest.json").resolve():
        raise ValueError(f"{dataset}: summary가 선택하지 않은 source run을 참조합니다.")

    output_files = payload.get("output_files", {})
    if set(output_files) != {"compression_summary.csv", "retrieval_summary.csv"}:
        raise ValueError(f"{dataset}: summary output file contract mismatch")
    for name, metadata in output_files.items():
        path = summary_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        if path.stat().st_size != int(metadata["bytes"]):
            raise ValueError(f"{dataset}/{name}: byte-size mismatch")
        if sha256_file(path) != metadata["sha256"]:
            raise ValueError(f"{dataset}/{name}: SHA-256 mismatch")

    if VERIFY_SOURCE_SHA256:
        for name, metadata in payload.get("source_files", {}).items():
            path = resolve_manifest_path(str(metadata["path"]))
            if path.stat().st_size != int(metadata["bytes"]):
                raise ValueError(f"{dataset}/{name}: source byte-size mismatch")
            if sha256_file(path) != metadata["sha256"]:
                raise ValueError(f"{dataset}/{name}: source SHA-256 mismatch")
    return payload


SUMMARY_DIRS = {}
SUMMARY_MANIFESTS = {}
generation_rows = []
for dataset, run_dir in SELECTED_RUNS.items():
    summary_dir = (
        PROJECT_ROOT
        / "results"
        / "paper"
        / dataset
        / SELECTED_RUN_IDS[dataset]
        / "summaries"
    )
    SUMMARY_DIRS[dataset] = summary_dir
    expected_outputs = tuple(
        summary_dir / name
        for name in (
            "compression_summary.csv",
            "retrieval_summary.csv",
            "summary_manifest.json",
        )
    )
    present_count = sum(path.exists() for path in expected_outputs)
    should_generate = REBUILD_DATASET_SUMMARIES or present_count == 0
    if 0 < present_count < len(expected_outputs) and not REBUILD_DATASET_SUMMARIES:
        raise RuntimeError(
            f"{dataset}: 일부 summary만 존재합니다. 원인을 확인한 뒤 "
            "REBUILD_DATASET_SUMMARIES=True로 재생성하십시오."
        )
    if should_generate:
        result = generate(
            run_dir,
            summary_dir,
            chunksize=CHUNKSIZE,
            overwrite=REBUILD_DATASET_SUMMARIES,
        )
        action = "generated"
    else:
        result = {"output_dir": str(summary_dir)}
        action = "reused"
    payload = validate_summary_bundle(
        summary_dir,
        dataset=dataset,
        run_dir=run_dir,
    )
    SUMMARY_MANIFESTS[dataset] = payload
    generation_rows.append(
        {
            "dataset": dataset,
            "action": action,
            "run_id": payload["run_id"],
            "model_uid": payload["model_uid"],
            **payload["validated_counts"],
            "summary_dir": summary_dir.relative_to(PROJECT_ROOT).as_posix(),
        }
    )

generation_table = pd.DataFrame(generation_rows)
display(generation_table)


## 2. 두 summary의 schema·값 범위 검증과 결합

결합 grain은 dataset/model/run/compression profile/threshold policy당 1행입니다. 압축 오차와 검색 지표는 서로 다른 grain이므로 명시적 provenance key와 `many_to_one` 검증 없이 합치지 않습니다.


In [ ]:
compression_required = {
    "dataset", "model_uid", "run_id", "extraction_uid",
    "origin_embedding_artifact_uid", "compression_family",
    "compression_profile", "sample_count", "mean_angular_error_rad",
    "p95_angular_error_rad", "mean_cosine_to_origin",
    "storage_bytes_per_embedding", "codebook_bytes",
    "codebook_bytes_source", "origin_fallback_count",
}
retrieval_required = {
    "dataset", "model_uid", "run_id", "extraction_uid",
    "origin_embedding_artifact_uid", "protocol_uid",
    "compression_family", "compression_profile", "threshold_policy",
    "query_count", "mated_count", "non_mated_count",
    "origin_dir_rank1", "compressed_dir_rank1",
    "origin_fpir", "compressed_fpir", "agreement_with_origin_rate",
    "threshold_crossing_rate", "storage_bytes_per_embedding",
    "codebook_bytes", "codebook_bytes_source", "origin_fallback_count",
}
compression_frames = []
retrieval_frames = []
for dataset, summary_dir in SUMMARY_DIRS.items():
    compression = pd.read_csv(summary_dir / "compression_summary.csv")
    retrieval = pd.read_csv(summary_dir / "retrieval_summary.csv")
    missing_compression = sorted(compression_required - set(compression.columns))
    missing_retrieval = sorted(retrieval_required - set(retrieval.columns))
    if missing_compression or missing_retrieval:
        raise ValueError(
            f"{dataset}: missing columns: "
            f"compression={missing_compression}, retrieval={missing_retrieval}"
        )
    if not compression["dataset"].eq(dataset).all() or not retrieval["dataset"].eq(dataset).all():
        raise ValueError(f"{dataset}: dataset column mismatch")
    if not compression["model_uid"].eq(MODEL_UIDS[dataset]).all() or not retrieval["model_uid"].eq(MODEL_UIDS[dataset]).all():
        raise ValueError(f"{dataset}: model_uid column mismatch")
    if not compression["run_id"].eq(SELECTED_RUN_IDS[dataset]).all() or not retrieval["run_id"].eq(SELECTED_RUN_IDS[dataset]).all():
        raise ValueError(f"{dataset}: run_id column mismatch")
    if (compression["origin_fallback_count"] != 0).any() or (retrieval["origin_fallback_count"] != 0).any():
        raise ValueError(f"{dataset}: origin fallback row was detected")

    for column in (
        "origin_dir_rank1", "compressed_dir_rank1", "origin_fpir",
        "compressed_fpir", "agreement_with_origin_rate",
        "threshold_crossing_rate",
    ):
        if not retrieval[column].dropna().between(0.0, 1.0).all():
            raise ValueError(f"{dataset}/{column}: value outside [0, 1]")
    if (compression["mean_angular_error_rad"].dropna() < 0).any():
        raise ValueError(f"{dataset}: negative angular error")

    compression_key = [
        "dataset", "model_uid", "run_id", "compression_family",
        "compression_profile",
    ]
    retrieval_key = [*compression_key, "threshold_policy"]
    if compression.duplicated(compression_key).any():
        raise ValueError(f"{dataset}: duplicate compression summary key")
    if retrieval.duplicated(retrieval_key).any():
        raise ValueError(f"{dataset}: duplicate retrieval summary key")
    if set(compression["compression_profile"]) != set(retrieval["compression_profile"]):
        raise ValueError(f"{dataset}: compression/retrieval profile set mismatch")
    compression_frames.append(compression)
    retrieval_frames.append(retrieval)

compression_profile_all = pd.concat(compression_frames, ignore_index=True)
retrieval_summary_all = pd.concat(retrieval_frames, ignore_index=True)

join_keys = [
    "dataset", "model_uid", "run_id", "extraction_uid",
    "origin_embedding_artifact_uid", "compression_family",
    "compression_profile", "storage_bytes_per_embedding",
    "codebook_bytes", "codebook_bytes_source",
]
compression_for_join = compression_profile_all.rename(
    columns={"origin_fallback_count": "compression_origin_fallback_count"}
)
retrieval_for_join = retrieval_summary_all.rename(
    columns={"origin_fallback_count": "retrieval_origin_fallback_count"}
)
compression_summary_all = compression_for_join.merge(
    retrieval_for_join,
    on=join_keys,
    how="inner",
    validate="one_to_many",
).sort_values(
    [
        "dataset", "threshold_policy", "compression_family",
        "storage_bytes_per_embedding", "compression_profile",
    ]
).reset_index(drop=True)

if len(compression_summary_all) != len(retrieval_summary_all):
    raise ValueError("summary join dropped or duplicated retrieval rows")
display(
    compression_summary_all[
        [
            "dataset", "model_uid", "run_id", "compression_family",
            "compression_profile", "threshold_policy",
            "storage_bytes_per_embedding", "mean_angular_error_rad",
            "compressed_dir_rank1", "compressed_fpir",
            "agreement_with_origin_rate", "threshold_crossing_rate",
        ]
    ]
)


## 3. 저장량–왜곡 및 open-set 성능 시각화

LFW와 SurvFace는 protocol과 목표 FPIR가 다르므로 데이터셋별 panel로 표시합니다. PQ codebook bytes는 payload와 별도 열에 보존되므로 x축만으로 총 저장량을 단정하지 않습니다.


In [ ]:
import matplotlib.pyplot as plt

figures = {}
dataset_order = list(DATASETS)

fig_distortion, axes = plt.subplots(
    1, len(dataset_order), figsize=(6 * len(dataset_order), 4.5), squeeze=False
)
geometry = compression_summary_all.drop_duplicates(
    ["dataset", "compression_family", "compression_profile"]
)
for axis, dataset in zip(axes[0], dataset_order):
    subset = geometry.loc[geometry["dataset"].eq(dataset)]
    for family, group in subset.groupby("compression_family", sort=True):
        ordered = group.sort_values("storage_bytes_per_embedding")
        axis.plot(
            ordered["storage_bytes_per_embedding"],
            ordered["mean_angular_error_rad"],
            marker="o",
            label=family.upper(),
        )
    axis.set_xscale("log", base=2)
    axis.set(
        title=f"{dataset.upper()}: storage vs distortion",
        xlabel="Representation payload (bytes / embedding)",
        ylabel="Mean angular error (rad)",
    )
    axis.grid(alpha=0.25)
    axis.legend()
fig_distortion.tight_layout()
figures["storage_distortion"] = fig_distortion

metrics = (
    ("compressed_dir_rank1", "Compressed DIR rank-1"),
    ("compressed_fpir", "Compressed FPIR"),
)
fig_open_set, axes = plt.subplots(
    len(metrics), len(dataset_order),
    figsize=(6 * len(dataset_order), 4.2 * len(metrics)),
    squeeze=False,
)
for column_index, dataset in enumerate(dataset_order):
    subset = compression_summary_all.loc[
        compression_summary_all["dataset"].eq(dataset)
    ]
    for row_index, (metric, ylabel) in enumerate(metrics):
        axis = axes[row_index, column_index]
        for (family, policy), group in subset.groupby(
            ["compression_family", "threshold_policy"], sort=True
        ):
            ordered = group.sort_values("storage_bytes_per_embedding")
            axis.plot(
                ordered["storage_bytes_per_embedding"],
                ordered[metric],
                marker="o",
                label=f"{family.upper()} / {policy}",
            )
        axis.set_xscale("log", base=2)
        axis.set(
            title=f"{dataset.upper()}: {ylabel}",
            xlabel="Representation payload (bytes / embedding)",
            ylabel=ylabel,
        )
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)
fig_open_set.tight_layout()
figures["storage_open_set"] = fig_open_set


## 4. 공통 CSV·그림·manifest 저장

공통 출력에는 compact table과 그림만 기록합니다. `compression_summary_all.csv`는 압축 profile 지표와 threshold-policy별 retrieval 지표를 함께 포함하며, source summary manifest의 SHA-256을 별도 manifest에 기록합니다.


In [ ]:
from datetime import datetime, timezone

selection_tag = "__".join(
    f"{dataset}-{SELECTED_RUN_IDS[dataset]}" for dataset in DATASETS
)
COMMON_OUTPUT_DIR = (
    PROJECT_ROOT / "results" / "paper" / "common" / MODEL_NAME / selection_tag
)
export_result = {
    "status": "computed_not_written",
    "destination": COMMON_OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix(),
    "compression_summary_all_rows": int(len(compression_summary_all)),
}

if WRITE_OUTPUTS:
    COMMON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_paths = {
        "compression_summary_all.csv": COMMON_OUTPUT_DIR / "compression_summary_all.csv",
        "retrieval_summary_all.csv": COMMON_OUTPUT_DIR / "retrieval_summary_all.csv",
        "storage_distortion.png": COMMON_OUTPUT_DIR / "storage_distortion.png",
        "storage_open_set.png": COMMON_OUTPUT_DIR / "storage_open_set.png",
    }
    existing = [path for path in output_paths.values() if path.exists()]
    manifest_path = COMMON_OUTPUT_DIR / "cross_dataset_summary_manifest.json"
    if manifest_path.exists():
        existing.append(manifest_path)
    if existing and not OVERWRITE_COMMON_OUTPUTS:
        raise FileExistsError(
            f"공통 출력이 이미 있습니다. OVERWRITE_COMMON_OUTPUTS=True가 필요합니다: {existing}"
        )

    compression_summary_all.to_csv(
        output_paths["compression_summary_all.csv"],
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        float_format="%.12g",
    )
    retrieval_summary_all.to_csv(
        output_paths["retrieval_summary_all.csv"],
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        float_format="%.12g",
    )
    figures["storage_distortion"].savefig(
        output_paths["storage_distortion.png"], dpi=180, bbox_inches="tight"
    )
    figures["storage_open_set"].savefig(
        output_paths["storage_open_set.png"], dpi=180, bbox_inches="tight"
    )

    common_manifest = {
        "schema_version": 1,
        "artifact_type": "cross_dataset_compact_summary",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_name": MODEL_NAME,
        "datasets": list(DATASETS),
        "selected_runs": {
            dataset: {
                "run_id": SELECTED_RUN_IDS[dataset],
                "model_uid": MODEL_UIDS[dataset],
                "summary_manifest": {
                    "path": (
                        SUMMARY_DIRS[dataset] / "summary_manifest.json"
                    ).relative_to(PROJECT_ROOT).as_posix(),
                    "sha256": sha256_file(
                        SUMMARY_DIRS[dataset] / "summary_manifest.json"
                    ),
                },
            }
            for dataset in DATASETS
        },
        "join_contract": {
            "grain": (
                "one row per dataset, model, run, compression profile, "
                "and threshold policy"
            ),
            "validation": "compression one-to-retrieval many",
            "rows": int(len(compression_summary_all)),
        },
        "output_files": {
            name: {
                "path": path.relative_to(PROJECT_ROOT).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for name, path in output_paths.items()
        },
    }
    manifest_path.write_text(
        json.dumps(common_manifest, ensure_ascii=False, indent=2, sort_keys=True)
        + "\n",
        encoding="utf-8",
        newline="\n",
    )
    export_result = {
        "status": "written",
        "destination": COMMON_OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix(),
        "manifest": manifest_path.relative_to(PROJECT_ROOT).as_posix(),
        "compression_summary_all_rows": int(len(compression_summary_all)),
        "retrieval_summary_all_rows": int(len(retrieval_summary_all)),
    }
export_result


## Final check

- `MODEL_UIDS`와 실제 workflow `RUN_IDS`를 혼동하지 않습니다.
- `origin_fallback_count`가 0이 아닌 summary는 결합하지 않습니다.
- `compression_summary_all.csv`는 compact summary만 포함하며 대용량 query/sample ledger를 복제하지 않습니다.
- LFW와 SurvFace의 protocol·목표 FPIR 차이 때문에 절대 DIR/FPIR를 동일 난이도로 직접 비교하지 않습니다.
- `VERIFY_SOURCE_SHA256=True`는 수 GB 원본 전체를 다시 읽으므로 최종 보존 검증 시에만 사용합니다.
